#### Feature Engineering

load data

In [2]:
import pandas as pd
from pathlib import Path

# Load cleaned data
data_path = Path("../data/movies_clean.csv")   
df = pd.read_csv(data_path)

df.head()

,movie_id,movie_title,movie_info,rating,genre,directors,in_theaters_date,on_streaming_date,runtime_in_minutes,critic_rating,critic_count,audience_rating,audience_count,release_year
0,1,Percy Jackson & the Olympians: The Lightning T...,A teenager discovers he's the descendant of a ...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,2010-02-12,2010-06-29,83.0,49,144,53.0,254287.0,2010
1,2,Please Give,Kate has a lot on her mind. There's the ethics...,R,Comedy,Nicole Holofcener,2010-04-30,2010-10-19,90.0,86,140,64.0,11567.0,2010
2,3,10,Blake Edwards' 10 stars Dudley Moore as George...,R,"Comedy, Romance",Blake Edwards,1979-10-05,1997-08-27,118.0,68,22,53.0,14670.0,1979
3,6,"10,000 B.C.",A young outcast from a primitive tribe is forc...,PG-13,"Action & Adventure, Classics, Drama",Roland Emmerich,2008-03-07,2008-06-24,109.0,8,148,37.0,411087.0,2008
4,9,Charly (A Heartbeat Away),Sam Roberts thinks he has all the answers: the...,PG,"Comedy, Drama, Romance",Adam Thomas Anderegg,2002-09-27,2003-08-01,103.0,20,10,87.0,4819.0,2002


In [3]:
df.shape

(14294, 14)

Split into training (before 2010) and test (2010 and after)

In [4]:
# Training: movies released before 2010
train_df = df[df["release_year"] < 2010].copy()

# Test: movies released in 2010 or later
test_df = df[df["release_year"] >= 2010].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (8204, 14)
Test shape: (6090, 14)


To predict the critic rating before any reviews are posted, features must only use information available at or before release (title, genre, runtime, release year, etc.). Columns that directly reflect post-release reception(critic_rating, critic_count, audience_rating, and audience_count) would leak future information into the model and therefore should not be used as predictors.

In [5]:
# columns that NOT be used as predictors
leak_cols = ["critic_rating", "critic_count", "audience_rating", "audience_count"]
# target variable
y_train = train_df["critic_rating"].copy()
y_test  = test_df["critic_rating"].copy()
# feature sets
X_train = train_df.drop(columns=leak_cols)
X_test  = test_df.drop(columns=leak_cols)

Using only the training data, create a new DataFrame containing the following ID
column and features:

• movie_title

• runtime_in_minutes

• NEW: kid_friendly (1 if G or PG, 0 if other ratings)

• NEW: dummy variable columns for each genre

In [6]:
feat_train = train_df[["movie_title", "runtime_in_minutes", "rating", "genre"]].copy()

# kid_friendly: 1 if rating is G or PG, otherwise 0
feat_train["kid_friendly"] = feat_train["rating"].isin(["G", "PG"]).astype(int)

# genre dummies (multi-genre strings split on commas)
genre_dummies = (
    feat_train["genre"]
        .str.get_dummies(sep=",")              # one column per genre token
        .rename(columns=lambda x: x.strip())   # strip spaces around names
)

# drop any duplicate dummy columns (keep the first copy)
genre_dummies = genre_dummies.loc[:, ~genre_dummies.columns.duplicated()]

# combine into final feature DataFrame
feat_train = pd.concat(
    [
        feat_train[["movie_title", "runtime_in_minutes", "kid_friendly"]],
        genre_dummies,
    ],
    axis=1,
)

feat_train.head()

,movie_title,runtime_in_minutes,kid_friendly,Animation,Anime & Manga,Art House & International,Classics,Comedy,Cult Movies,Documentary,...,Kids & Family,Musical & Performing Arts,Mystery & Suspense,Romance,Science Fiction & Fantasy,Special Interest,Sports & Fitness,Television,Western,Action & Adventure
2,10,118.0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,"10,000 B.C.",109.0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,Charly (A Heartbeat Away),103.0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5,The Adventures of Mark Twain,86.0,1,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
6,Dark Water,100.0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


Create 3 new features

kid_family:

flags movies that are both G/PG and labeled as Kids & Family, capturing a very specific family-oriented segment where critics may score films differently than general-audience titles.

In [7]:
# kid_family: kid-friendly AND tagged as Kids & Family
genre_cols = [c for c in feat_train.columns 
              if c not in ["movie_title", "runtime_in_minutes", "kid_friendly"]]

if "Kids & Family" in genre_cols:
    feat_train["kid_family"] = feat_train["kid_friendly"] * feat_train["Kids & Family"]
else:
    # fall back if that exact column name isn't present
    feat_train["kid_family"] = 0

days_to_stream:

measures how long a movie stayed in theatres before hitting streaming. Longer gaps may correspond to bigger or more prestigious releases, which could be associated with different critic scores.

In [8]:
# make sure dates are datetime
train_df["in_theaters_date"] = pd.to_datetime(train_df["in_theaters_date"], errors="coerce")
train_df["on_streaming_date"] = pd.to_datetime(train_df["on_streaming_date"], errors="coerce")

# days_to_stream: days between theatrical and streaming releases
feat_train["days_to_stream"] = (
    (train_df["on_streaming_date"] - train_df["in_theaters_date"])
    .dt.days
)

holiday_blockbuster:

flags movies released during the holiday season (November–December) that are in major blockbuster genres (Action & Adventure, Sci-Fi & Fantasy, or Mystery & Suspense). These tentpole holiday releases are marketed differently and may be reviewed differently by critics than non-holiday or non-blockbuster titles.

In [9]:
# holiday months: November and December
holiday_month = train_df["in_theaters_date"].dt.month.isin([11, 12])

# some big "blockbuster" genres (only use ones that actually exist)
blockbuster_genres = [
    g for g in ["Action & Adventure", "Science Fiction & Fantasy", "Mystery & Suspense"]
    if g in feat_train.columns
]

# combine those genres into a single indicator
if blockbuster_genres:
    blockbuster_flag = feat_train[blockbuster_genres].max(axis=1) == 1
else:
    blockbuster_flag = False

# holiday_blockbuster: holiday release AND in at least one blockbuster genre
feat_train["holiday_blockbuster"] = (holiday_month & blockbuster_flag).astype(int)

feat_train["holiday_blockbuster"].value_counts()

holiday_blockbuster
0    7656
1     548
Name: count, dtype: int64

#### Apply the transformations

Function for feature engineering

In [10]:
# function to do all feature engineering steps
def make_feature_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Given a movies DataFrame, return a feature DataFrame with:
      - movie_title  
      - runtime_in_minutes
      - kid_friendly         (1 if rating is G/PG, else 0)
      - one-hot genre dummies
      - kid_family           (kid_friendly AND Kids & Family)
      - days_to_stream       (days from theatrical to streaming release)
      - holiday_blockbuster  (holiday release in a blockbuster genre)
    """
    # work on a copy
    feat = df[[
        "movie_title",
        "runtime_in_minutes",
        "rating",
        "genre",
        "in_theaters_date",
        "on_streaming_date",
    ]].copy()

    # kid_friendly feature
    feat["kid_friendly"] = feat["rating"].isin(["G", "PG"]).astype(int)

    # genre dummies (multi-genre strings split on commas) 
    genre_dummies = (
        feat["genre"]
        .str.get_dummies(sep=",")
        .rename(columns=lambda x: x.strip())
    )
    # drop duplicate dummy columns if they exist
    genre_dummies = genre_dummies.loc[:, ~genre_dummies.columns.duplicated()]

    # base feature set
    feat_num = pd.concat(
        [feat[["movie_title", "runtime_in_minutes", "kid_friendly"]], genre_dummies],
        axis=1,
    )

    # new feature 1: kid_family (kid-friendly AND Kids & Family genre)
    if "Kids & Family" in genre_dummies.columns:
        feat_num["kid_family"] = (
            feat_num["kid_friendly"] * genre_dummies["Kids & Family"]
        )
    else:
        feat_num["kid_family"] = 0

    # dates for later features 
    dates = feat[["in_theaters_date", "on_streaming_date"]].apply(
        pd.to_datetime, errors="coerce"
    )

    # new feature 2: days_to_stream 
    feat_num["days_to_stream"] = (
        dates["on_streaming_date"] - dates["in_theaters_date"]
    ).dt.days

    # new feature 3: holiday_blockbuster 
    # holiday season = November or December
    holiday_month = dates["in_theaters_date"].dt.month.isin([11, 12])

    # “blockbuster” genres we care about (only keep ones that exist)
    blockbuster_genres = [
        g
        for g in ["Action & Adventure", "Science Fiction & Fantasy", "Mystery & Suspense"]
        if g in genre_dummies.columns
    ]
    if blockbuster_genres:
        blockbuster_flag = genre_dummies[blockbuster_genres].max(axis=1) == 1
    else:
        blockbuster_flag = False

    feat_num["holiday_blockbuster"] = (holiday_month & blockbuster_flag).astype(int)

    return feat_num

Apply the same transformations to train and test

In [11]:
# y: critic rating 
y_train = train_df["critic_rating"].astype(float).copy()
y_test  = test_df["critic_rating"].astype(float).copy()

# X: apply exactly the same feature logic to train and test
feat_train = make_feature_df(train_df)
feat_test  = make_feature_df(test_df)

# drop non-numeric ID column for modeling
X_train = feat_train.drop(columns=["movie_title"]).copy()
X_test  = feat_test.drop(columns=["movie_title"]).copy()

# make sure everything is numeric
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test  = X_test.apply(pd.to_numeric, errors="coerce")

# align test columns to train columns (important if some genres only appear in one split)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

All feature engineering steps are wrapped in a single function make_feature_df, which is called on both train_df and test_df. This guarantees that X_train and X_test are built using the exact same logic (same dummy-encoding of genres, and the same three engineered features: kid_family, days_to_stream, and holiday_blockbuster). Afterward, I reindex X_test to the columns of X_train so that both sets have an identical set of predictors.

In [12]:
# Impute missing values: 0 for dummy vars, median for other numeric vars
for col in X_train.columns:
    unique_vals = set(X_train[col].dropna().unique())

    if unique_vals.issubset({0, 1}):
        # one-hot / dummy column: keep it strictly 0/1
        fill_val = 0
    else:
        # other numeric column: use training median
        fill_val = X_train[col].median()

    X_train[col] = X_train[col].fillna(fill_val)
    X_test[col]  = X_test[col].fillna(fill_val)   # use same fill_val as train

In [13]:
print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0


For modeling, I drop the string ID column movie_title and keep only numeric features. For one-hot/dummy variables (0/1 columns), I impute missing values with 0, interpreting missing as “category not present.” For all other numeric columns, I impute missing values using the median computed from the training data, and apply the same median to the test set to avoid leaking information from test into training. This keeps dummy variables strictly 0/1 and uses a simple, robust strategy for continuous features, while allowing me to retain all rows instead of dropping observations with missing data. After these steps, X_train, y_train, X_test, and y_test contain only numeric, non-null values and are ready for fitting predictive models.

#### Save data

In [14]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8204, 26) (6090, 26) (8204,) (6090,)


In [15]:
X_train.to_csv("../data/X_train_movies.csv", index=False)
y_train.to_csv("../data/y_train_movies.csv", index=False)

X_test.to_csv("../data/X_test_movies.csv", index=False)
y_test.to_csv("../data/y_test_movies.csv", index=False)